# 🧥 ARVTON — AR/VR Virtual Try-On · Full Training Notebook

> **Run this notebook top-to-bottom on Google Colab (T4 free tier or A100 Pro)**

## What this notebook does
| Step | Description | Est. Time (T4) |
|------|-------------|----------------|
| 1 | GPU check + environment setup | ~2 min |
| 2 | Clone repo + install dependencies | ~5 min |
| 3 | Mount Google Drive | ~1 min |
| 4 | **Phase 0** — Full dataset pipeline (ViViD + CC0 + Synthetic) | ~30-90 min |
| 5 | Configure training hyperparameters | instant |
| 6 | Build dataset loaders | ~1 min |
| 7 | Build models | ~1 min |
| 8 | Setup optimizers / loss / AMP | instant |
| 9 | **Train GAN model** | ~180-240 min |
| 10 | Plot loss curves + save to Drive | ~2 min |

---
### Architecture Overview
```
Person Photo + Garment Image
        │
     [SAM 2] → Segmentation masks
  [Leffa / CatVTON] → 2D try-on composite
  [HMR 2.0 + ECON] → SMPL mesh
  [TripoSR / SyncHuman] → .glb 3D model
  [RefineGenerator GAN] ← Fine-tuned here
```
**License:** MIT — Only CC0 / Apache 2.0 / MIT data & weights used.

---
## Step 1 — Check GPU & Python environment

In [1]:
# ─── GPU & CUDA check ────────────────────────────────────────────────
!nvidia-smi
import sys, torch

print(f"\nPython  : {sys.version.split()[0]}")
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.version.cuda}")
print(f"CUDA OK : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    vram  = props.total_memory / 1024**3
    print(f"GPU     : {props.name}")
    print(f"VRAM    : {vram:.1f} GB")
    if vram < 12:
        print("⚠️  Low VRAM — use BATCH_SIZE=1 and IMAGE_SIZE=(384,512)")
    elif vram < 20:
        print("✅ T4 detected — recommended settings pre-loaded below")
    else:
        print("🚀 A100/H100 detected — can use larger batch & resolution")
else:
    print("❌ No GPU found — go to Runtime → Change runtime type → T4 GPU")

Fri Mar  6 13:49:30 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

---
## Step 2 — Clone repository & install dependencies

In [2]:
import os

REPO_URL  = "https://github.com/Gandharv2323/AR-PROTOTYPE-4.git"
REPO_DIR  = "/content/AR-PROTOTYPE-4"
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
    print("✅ Repo cloned")
else:
    !cd {REPO_DIR} && git pull --quiet
    print("✅ Repo up-to-date")
%cd {REPO_DIR}
!ls -la

✅ Repo up-to-date
/content/AR-PROTOTYPE-4
total 1396
drwxr-xr-x 11 root root    4096 Mar  6 13:44  .
drwxr-xr-x  1 root root    4096 Mar  6 13:45  ..
-rw-r--r--  1 root root       0 Mar  6 13:44 '=0.10.0'
-rw-r--r--  1 root root     630 Mar  6 13:44 '=0.22.0'
-rw-r--r--  1 root root       0 Mar  6 13:44 '=0.27.0'
-rw-r--r--  1 root root       0 Mar  6 13:44 '=0.29.0'
-rw-r--r--  1 root root       0 Mar  6 13:44 '=10.3.0'
-rw-r--r--  1 root root       0 Mar  6 13:44 '=2.24.0'
-rw-r--r--  1 root root       0 Mar  6 13:44 '=4.40.0'
-rw-r--r--  1 root root       0 Mar  6 13:44 '=4.9.0'
drwxr-xr-x  5 root root    4096 Mar  6 13:43  app
drwxr-xr-x  5 root root    4096 Mar  6 13:43  arvton_app
-rw-r--r--  1 root root 1244802 Mar  6 13:43  ARVTON_Colab_Training.ipynb
-rw-r--r--  1 root root   37784 Mar  6 13:43  ARVTON_Final_Master_Prompt.md
-rw-r--r--  1 root root   12144 Mar  6 13:43  ARVTON_FineTune.ipynb
drwxr-xr-x  2 root root    4096 Mar  6 13:43  configs
drwxr-xr-x  3 root root    4096 

In [3]:
# Install all ARVTON dependencies (T4-compatible, skip heavy 3D libs for training only)
# Note: torch/torchvision already pre-installed in Colab; only install missing extras
!pip install -q \
    peft>=0.10.0 \
    accelerate>=0.29.0 \
    diffusers>=0.27.0 \
    transformers>=4.40.0 \
    open-clip-torch>=2.24.0 \
    opencv-python-headless>=4.9.0 \
    Pillow>=10.3.0 \
    scikit-image>=0.22.0 \
    numpy scipy tqdm matplotlib \
    yt-dlp pyyaml requests fastapi uvicorn python-multipart \
    gdown

print("✅ Core dependencies installed")


✅ Core dependencies installed


In [4]:
# Run the project setup script (creates dirs, verifies imports)
!python setup_local.py
print("✅ Project directories created")


  ARVTON -- Local Environment Setup

[1/5] Creating directories...
  [OK] 0 directories created, 12 already existed

[2/5] Creating sample manifests...
  [OK] Train manifest already exists at /content/AR-PROTOTYPE-4/data/arvton/datasets/train_manifest.json

[3/5] Checking Python...
  Python: 3.12.12
  [OK] Python version OK

[4/5] Checking GPU...
  PyTorch: 2.10.0+cu128
  [OK] GPU: Tesla T4
  [OK] VRAM: 14.6 GB
  [OK] CUDA: 12.8
  [OK] VRAM is sufficient for inference

[5/5] Checking dependencies...
  [OK] 7/8 core packages installed
  [FAIL] Missing: Trimesh (3D meshes)
    Run: pip install -r requirements.txt

  Next Steps

  1. Install dependencies (if not done):
     pip install -r requirements.txt

  2. Download model checkpoints:
     - SAM2 (segmentation):
       pip install segment-anything-2
       (auto-downloads from HuggingFace on first run)

     - Leffa (virtual try-on):
       pip install leffa
       (auto-downloads from HuggingFace on first run)

     - TripoSR (3D re

---
## Step 3 — Mount Google Drive

All checkpoints will be saved to `MyDrive/ARVTON/checkpoints/` so they **persist after the Colab session ends**.

In [5]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')
DRIVE_CKPT  = Path("/content/drive/MyDrive/ARVTON/checkpoints")
DRIVE_DATA  = Path("/content/drive/MyDrive/ARVTON/datasets")
DRIVE_LOGS  = Path("/content/drive/MyDrive/ARVTON/logs")
for d in [DRIVE_CKPT, DRIVE_DATA, DRIVE_LOGS]:
    d.mkdir(parents=True, exist_ok=True)
print(f"✅ Checkpoints → {DRIVE_CKPT}")
print(f"✅ Datasets    → {DRIVE_DATA}")
print(f"✅ Logs        → {DRIVE_LOGS}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Checkpoints → /content/drive/MyDrive/ARVTON/checkpoints
✅ Datasets    → /content/drive/MyDrive/ARVTON/datasets
✅ Logs        → /content/drive/MyDrive/ARVTON/logs


---
## Step 4 — Phase 0: Full Dataset Pipeline

This runs **all 4 dataset pipeline stages** in sequence:
1. **ViViD** — extracts ~9,700 person/garment frames from video dataset
2. **CC0 Scraper** — downloads ~2,000 CC0-licensed fashion images via Pexels API
3. **Synthetic Gen** — generates ~500 synthetic try-on pairs using diffusion models
4. **Validate & Manifest** — validates all images and builds the master training manifest

> ⚠️ **ViViD** requires GitHub access token to clone the dataset repo. If it fails, the scraper and synthetic stages still run.
>
> 💡 **API Keys** (optional but recommended for more data):
> - Get a free Pexels API key at https://www.pexels.com/api/
> - Get a free Pixabay API key at https://pixabay.com/api/docs/

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 🚀 PHASE 0: FULL DATASET GENERATION TO GOOGLE DRIVE (2TB READY)
# ══════════════════════════════════════════════════════════════════════
import os
import sys

# ── 1. Configure API Keys (optional — for Pexels/Pixabay extra data) ──
# Replace with your own keys for best results
os.environ['PEXELS_API_KEY']  = ''
os.environ['PIXABAY_API_KEY'] = ''  # Optional — leave blank to skip Pixabay

# ── 2. Set Drive paths so pipelines write directly to Drive ──────────
os.environ['ARVTON_DRIVE_DATA']  = '/content/drive/MyDrive/ARVTON/datasets'
os.environ['ARVTON_DRIVE_CKPT']  = '/content/drive/MyDrive/ARVTON/checkpoints'
os.environ['ARVTON_DRIVE_LOGS']  = '/content/drive/MyDrive/ARVTON/logs'

# ── 3. Make sure we are in the project root ──────────────────────────
%cd /content/AR-PROTOTYPE-4

# ── 4. Run all pipeline stages ───────────────────────────────────────
print('\n' + '='*60)
print('⚙️  1. STARTING ViViD DATASET PREPARATION (~9,700 images)...')
print('='*60)
!python -m pipeline.vivid_prepare 2>&1

print('\n' + '='*60)
print('⚙️  2. STARTING CC0 WEB SCRAPER (~2,000 images)...')
print('='*60)
!python -m pipeline.cc0_scraper 2>&1

print('\n' + '='*60)
print('⚙️  3. STARTING SYNTHETIC GENERATION (~500 pairs)...')
print('='*60)
!python -m pipeline.synthetic_gen 2>&1

print('\n' + '='*60)
print('⚙️  4. VALIDATING AND BUILDING MASTER MANIFEST...')
print('='*60)
!python -m pipeline.validate_dataset 2>&1

print('\n✅ FULL PHASE 0 DATASET COMPLETE!')
print(f'   Check Google Drive → MyDrive/ARVTON/datasets')

/content/AR-PROTOTYPE-4

⚙️  1. STARTING ViViD DATASET PREPARATION (~9,700 images)...
2026-03-06 13:50:04 | arvton.vivid_prepare | INFO | ============================================================
2026-03-06 13:50:04 | arvton.vivid_prepare | INFO | ARVTON — ViViD Dataset Preparation (Source A)
2026-03-06 13:50:04 | arvton.vivid_prepare | INFO | ============================================================
2026-03-06 13:50:04 | arvton.platform | INFO | Platform detected: Google Colab
2026-03-06 13:50:04 | arvton.platform | INFO | Platform detected: Google Colab
2026-03-06 13:50:04 | arvton.platform | ERROR | Failed to mount Google Drive: 'NoneType' object has no attribute 'kernel'
2026-03-06 13:50:06 | arvton.platform | INFO | GPU: Tesla T4 | Memory: 14.6 GB | ROCm: False | Precision: fp16
ARVTON System Information
  Python:    3.12.12
  OS:        Linux 6.6.113+
2026-03-06 13:50:06 | arvton.platform | INFO | Platform detected: Google Colab
  Platform:  colab
  PyTorch:   2.10.0+cu128


Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/AR-PROTOTYPE-4/pipeline/synthetic_gen.py", line 776, in <module>
    main()
  File "/content/AR-PROTOTYPE-4/pipeline/synthetic_gen.py", line 756, in main
    records = process_synthetic_pipeline(config, paths, device=device)
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/AR-PROTOTYPE-4/pipeline/synthetic_gen.py", line 550, in process_synthetic_pipeline
    sdxl_pipeline = load_sdxl_pipeline(device=device)
                    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/AR-PROTOTYPE-4/pipeline/synthetic_gen.py", line 64, in load_sdxl_pipeline
    from diffusers import StableDiffusionXLPipeline
  File "<frozen importlib._bootstrap>", line 1412, in _handle_fromlist
  File "/usr/local/lib/python3.12/dist-packages/diffusers/utils/import_utils.py", line 1007, in __getattr__
    value = getat

In [ ]:
# ── Verify dataset ────────────────────────────────────────────────────
import json
from pathlib import Path

# Check both local and Drive manifest locations
MANIFEST_PATHS = [
    Path("/content/drive/MyDrive/ARVTON/datasets/train_manifest.json"),
    Path("/content/AR-PROTOTYPE-4/data/arvton/datasets/train_manifest.json"),
]

manifest = None
for mp in MANIFEST_PATHS:
    if mp.exists():
        manifest = mp
        break

if manifest:
    raw = json.loads(manifest.read_text())
    records = raw if isinstance(raw, list) else raw.get("entries", [])
    print(f"✅ Manifest found: {len(records)} training pairs")
    print(f"   Location: {manifest}")
    if records:
        print("  Sample keys:", list(records[0].keys()))
else:
    print("⚠️  No manifest found — synthetic fallback will be used in training")

# Count image files in local dataset dir
data_root = Path("data/arvton/datasets")
if data_root.exists():
    persons   = list(data_root.rglob("person_*.jpg")) + list(data_root.rglob("person_*.png"))
    garments  = list(data_root.rglob("garment_*.jpg")) + list(data_root.rglob("garment_*.png"))
    masks     = list(data_root.rglob("mask_*.png"))
    print(f"  Person images  : {len(persons)}")
    print(f"  Garment images : {len(garments)}")
    print(f"  Mask images    : {len(masks)}")

---
## Step 5 — Configure Training Hyperparameters

Settings are **auto-tuned** for T4 (16 GB VRAM). Edit the override section below if needed.

In [ ]:
import torch

# ── Detect GPU tier ───────────────────────────────────────────────────
if torch.cuda.is_available():
    vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1024**3
    gpu_name = torch.cuda.get_device_name(0)
else:
    vram_gb  = 0
    gpu_name = "CPU"

# ── Auto-tune defaults based on VRAM ──────────────────────────────────
if vram_gb >= 35:          # A100 40 GB / H100
    BATCH_SIZE = 8
    IMAGE_SIZE = (768, 1024)
    LORA_RANK  = 32
    PRECISION  = "bf16"
elif vram_gb >= 13:        # T4 16 GB (free tier) — conservative settings
    BATCH_SIZE = 1         # Keep at 1 to avoid OOM with LoRA+AMP+512px
    IMAGE_SIZE = (384, 512)
    LORA_RANK  = 8
    PRECISION  = "fp16"
else:                      # 8 GB / CPU fallback
    BATCH_SIZE = 1
    IMAGE_SIZE = (256, 384)
    LORA_RANK  = 4
    PRECISION  = "fp16"

# ── Override any value here ────────────────────────────────────────────
EPOCHS              = 180      # Total training epochs
LR_GENERATOR        = 2e-4    # Generator LR  (AdamW)
LR_DISCRIMINATOR    = 2e-4    # Discriminator LR
USE_AMP             = True    # Mixed precision (FP16) — saves ~30% VRAM
USE_LORA            = True    # LoRA adapters — reduces trainable params <5%
SAVE_EVERY          = 20      # Checkpoint every N epochs (+ always on best val)
NUM_WORKERS         = 2       # DataLoader worker processes
VAL_SPLIT           = 0.08    # 8% of data held out for validation
GRAD_ACCUM_STEPS    = 4       # Effective batch = BATCH_SIZE × GRAD_ACCUM_STEPS
EMA_DECAY           = 0.999   # EMA decay rate
EARLY_STOP_PATIENCE = 20      # Stop if no improvement for N epochs
EARLY_STOP_MIN_DELTA= 1e-4    # Min improvement to count as progress
WARMUP_EPOCHS       = 5       # LR warmup epochs
USE_SPECTRAL_NORM   = True    # Spectral norm for discriminator stability
USE_WANDB           = False   # Weights & Biases logging (requires wandb login)
GRAD_CLIP_NORM      = 1.0     # Max L2 norm for gradient clipping

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
est_min = EPOCHS * 1.2

print("=" * 58)
print(f"  GPU            : {gpu_name}")
print(f"  VRAM           : {vram_gb:.1f} GB")
print(f"  Device         : {DEVICE}")
print("-" * 58)
print(f"  Epochs         : {EPOCHS}")
print(f"  Batch size     : {BATCH_SIZE}  (effective: {BATCH_SIZE * GRAD_ACCUM_STEPS})")
print(f"  Image size     : {IMAGE_SIZE}")
print(f"  LR (G / D)     : {LR_GENERATOR} / {LR_DISCRIMINATOR}")
print(f"  Mixed prec.    : {PRECISION if USE_AMP else 'disabled'}")
print(f"  LoRA           : {'ON  (rank=' + str(LORA_RANK) + ')' if USE_LORA else 'OFF'}")
print(f"  Grad accum.    : {GRAD_ACCUM_STEPS} steps")
print(f"  Grad clip norm : {GRAD_CLIP_NORM}")
print(f"  Val split      : {VAL_SPLIT * 100:.0f} %")
print(f"  Est. time (T4) : ~{est_min:.0f} min  (~{est_min / 60:.1f} hrs)")

---
## Step 6 — Build Dataset & DataLoaders

Uses **real data** from the manifest if available, otherwise falls back to a synthetic random dataset for testing the training loop.

In [ ]:
import json, os
import numpy as np
import torch
from torch.utils.data import DataLoader, random_split, Dataset
from pathlib import Path

# ── Import project modules ────────────────────────────────────────────
import sys
sys.path.insert(0, '/content/AR-PROTOTYPE-4')
from pipeline.platform_utils import setup_logging, get_paths

setup_logging("INFO")
paths = get_paths()

# ── Synthetic fallback dataset ────────────────────────────────────────
class SyntheticTryOnDataset(Dataset):
    """Generates random image tensors for smoke-testing the training loop."""
    def __init__(self, size=200, image_size=(384, 512)):
        self.size = size
        self.h, self.w = image_size[1], image_size[0]

    def __len__(self):
        return self.size

    def __getitem__(self, idx):
        def rand_img(c=3):
            return torch.rand(c, self.h, self.w) * 2 - 1
        return {
            "person":    rand_img(),
            "garment":   rand_img(),
            "gt":        rand_img(),
            "mask":      rand_img(1),
            "densepose": rand_img(),
            "category":  "upper",
        }

# ── Try real manifests (local then Drive) ─────────────────────────────
MANIFEST_CANDIDATES = [
    str(paths["train_manifest"]),
    "/content/drive/MyDrive/ARVTON/datasets/train_manifest.json",
]
real_count = 0
MANIFEST = None
for mc in MANIFEST_CANDIDATES:
    try:
        with open(mc, "r") as f:
            raw = json.load(f)
        entries = raw if isinstance(raw, list) else raw.get("entries", list(raw.values()))
        cnt = len([e for e in entries if isinstance(e, dict) and "person_image" in e])
        if cnt > 0:
            real_count = cnt
            MANIFEST = mc
            break
    except Exception:
        pass

print(f"Real manifest entries: {real_count}")

if real_count >= 2:
    from pipeline.fine_tune import TryOnDataset
    full_ds = TryOnDataset(manifest_path=MANIFEST, image_size=IMAGE_SIZE, augment=True)
    print(f"Using real dataset: {len(full_ds)} samples")
else:
    print("No real data found — using synthetic dataset (200 random samples)")
    full_ds = SyntheticTryOnDataset(size=200, image_size=IMAGE_SIZE)

val_size   = max(1, int(len(full_ds) * VAL_SPLIT))
train_size = len(full_ds) - val_size
train_ds, val_ds = random_split(full_ds, [train_size, val_size])

# num_workers=0 avoids multiprocessing issues in Colab
train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=0, pin_memory=True, drop_last=True,
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=0, pin_memory=True,
)

print(f"Dataset ready")
print(f"   Train: {train_size} samples ({len(train_loader)} batches/epoch)")
print(f"   Val  : {val_size} samples")

---
## Step 7 — Build GAN Models

Instantiates the `RefineGenerator` (U-Net) + `MultiScaleDiscriminator` and utility classes.

In [ ]:
import logging
from pipeline.fine_tune   import apply_lora
from pipeline.refine      import RefineGenerator, MultiScaleDiscriminator, TryOnCombinedLoss

logger = logging.getLogger("arvton.colab")

# ── Simple utility classes (inline) ──────────────────────────────────
class EarlyStopping:
    def __init__(self, patience=10, min_delta=0.0, mode="min"):
        self.patience  = patience
        self.min_delta = min_delta
        self.mode      = mode
        self.counter   = 0
        self.best      = float("inf") if mode == "min" else float("-inf")
        self.stop      = False

    def __call__(self, val):
        improved = (val < self.best - self.min_delta) if self.mode == "min" \
                   else (val > self.best + self.min_delta)
        if improved:
            self.best    = val
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.stop = True
        return self.stop


class ExponentialMovingAverage:
    def __init__(self, model, decay=0.999, device=None):
        self.decay  = decay
        self.shadow = {k: v.clone().to(device) for k, v in model.state_dict().items()}

    def update(self, model):
        for k, v in model.state_dict().items():
            self.shadow[k] = self.decay * self.shadow[k] + (1 - self.decay) * v.detach()

    def apply(self, model):
        model.load_state_dict(self.shadow)


# ── Build Generator + Discriminator ──────────────────────────────────
generator     = RefineGenerator(in_channels=6, out_channels=3).to(DEVICE)
discriminator = MultiScaleDiscriminator(in_channels=3).to(DEVICE)

# Apply LoRA if requested (reduces GPU memory significantly)
if USE_LORA:
    try:
        generator = apply_lora(generator, rank=LORA_RANK)
        print(f"   LoRA adapters applied (rank={LORA_RANK})")
    except Exception as e:
        print(f"   LoRA skipped ({e})")

# Enable gradient checkpointing to trade compute for VRAM
if hasattr(generator, 'gradient_checkpointing_enable'):
    generator.gradient_checkpointing_enable()
    print("   Gradient checkpointing: ON")

g_params = sum(p.numel() for p in generator.parameters()     if p.requires_grad)
d_params = sum(p.numel() for p in discriminator.parameters() if p.requires_grad)
total    = g_params + d_params
print(f"Models built")
print(f"   Generator params     : {g_params:,}")
print(f"   Discriminator params : {d_params:,}")
print(f"   Total trainable      : {total:,}")

early_stopping = EarlyStopping(
    patience=EARLY_STOP_PATIENCE,
    min_delta=EARLY_STOP_MIN_DELTA,
    mode="min",
)
ema_generator = ExponentialMovingAverage(generator, decay=EMA_DECAY, device=DEVICE)
print(f"   Early stopping : patience={EARLY_STOP_PATIENCE}")
print( "   EMA decay      : " + str(EMA_DECAY))

---
## Step 8 — Optimizers, Schedulers, Loss & AMP

In [ ]:
from pathlib import Path

# ── Optimizers ────────────────────────────────────────────────────────
opt_G = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, generator.parameters()),
    lr=LR_GENERATOR, weight_decay=0.01, betas=(0.5, 0.999)
)
opt_D = torch.optim.AdamW(
    discriminator.parameters(),
    lr=LR_DISCRIMINATOR, weight_decay=0.01, betas=(0.5, 0.999)
)

# ── Schedulers ────────────────────────────────────────────────────────
sched_G = torch.optim.lr_scheduler.CosineAnnealingLR(opt_G, T_max=EPOCHS, eta_min=1e-6)
sched_D = torch.optim.lr_scheduler.CosineAnnealingLR(opt_D, T_max=EPOCHS, eta_min=1e-6)

# ── Loss ──────────────────────────────────────────────────────────────
criterion = TryOnCombinedLoss(device=DEVICE)

# ── AMP scalers (T4 uses fp16) ────────────────────────────────────────
dtype    = torch.float16 if PRECISION == 'fp16' else torch.bfloat16
_use_amp = USE_AMP and DEVICE == 'cuda'
scaler_G = torch.amp.GradScaler('cuda') if _use_amp else None
scaler_D = torch.amp.GradScaler('cuda') if _use_amp else None

# ── Checkpoint dir ────────────────────────────────────────────────────
CKPT_DIR = Path("data/arvton/checkpoints/gan")
CKPT_DIR.mkdir(parents=True, exist_ok=True)

# ── Training history ──────────────────────────────────────────────────
history       = {"train_G": [], "train_D": [], "val_G": [], "epoch_times": []}
best_val_loss = float('inf')

print("Optimizers, schedulers and losses ready")
print(f"   AMP      : {'ON (' + PRECISION + ')' if _use_amp else 'OFF'}")
print(f"   Ckpt dir : {CKPT_DIR}")

---
## Step 9 — Train the ARVTON GAN Model

This trains the `RefineGenerator` (U-Net) + `MultiScaleDiscriminator` using:
- **L1 reconstruction loss** + **VGG perceptual loss** + **GAN adversarial loss**
- **LoRA adapters** (optional, reduces trainable params to < 5%)
- **Mixed precision** (FP16) + **gradient checkpointing**
- **Gradient accumulation** — effective batch = `BATCH_SIZE × GRAD_ACCUM_STEPS`
- **EMA** of generator weights for stable evaluation
- **Early stopping** + best-checkpoint saving

In [ ]:
import time, shutil
from tqdm.notebook import tqdm
import torch.nn as nn

print(f"\n{'='*58}")
print(f"  Starting training: {EPOCHS} epochs")
print(f"  Device : {DEVICE}  |  AMP : {_use_amp}  |  LoRA : {USE_LORA}")
print(f"  Batch  : {BATCH_SIZE}  |  Effective : {BATCH_SIZE * GRAD_ACCUM_STEPS}")
print(f"{'='*58}\n")

generator.train()
discriminator.train()

for epoch in range(EPOCHS):
    epoch_start = time.time()

    # ── Training loop ─────────────────────────────────────────────────
    total_loss_G = 0.0
    total_loss_D = 0.0
    n_batches    = 0

    generator.train()
    discriminator.train()

    opt_G.zero_grad()
    opt_D.zero_grad()

    for batch_idx, batch in enumerate(tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}", leave=False)):
        person    = batch["person"].to(DEVICE, non_blocking=True)
        garment   = batch["garment"].to(DEVICE, non_blocking=True)
        gt        = batch["gt"].to(DEVICE, non_blocking=True)
        mask      = batch["mask"].to(DEVICE, non_blocking=True)

        # Concatenate person + garment as 6-channel input
        x_in = torch.cat([person, garment], dim=1)  # (B, 6, H, W)

        is_last_accum = (batch_idx + 1) % GRAD_ACCUM_STEPS == 0 or \
                        (batch_idx + 1) == len(train_loader)

        # ── Train Discriminator ────────────────────────────────────────
        if _use_amp:
            with torch.amp.autocast('cuda', dtype=dtype):
                fake = generator(x_in).detach()
                loss_D = criterion.discriminator_loss(discriminator, gt, fake)
                loss_D = loss_D / GRAD_ACCUM_STEPS
            scaler_D.scale(loss_D).backward()
            if is_last_accum:
                scaler_D.unscale_(opt_D)
                nn.utils.clip_grad_norm_(discriminator.parameters(), GRAD_CLIP_NORM)
                scaler_D.step(opt_D)
                scaler_D.update()
                opt_D.zero_grad()
        else:
            fake = generator(x_in).detach()
            loss_D = criterion.discriminator_loss(discriminator, gt, fake) / GRAD_ACCUM_STEPS
            loss_D.backward()
            if is_last_accum:
                nn.utils.clip_grad_norm_(discriminator.parameters(), GRAD_CLIP_NORM)
                opt_D.step()
                opt_D.zero_grad()

        # ── Train Generator ───────────────────────────────────────────
        if _use_amp:
            with torch.amp.autocast('cuda', dtype=dtype):
                fake = generator(x_in)
                loss_G = criterion(fake, gt, discriminator) / GRAD_ACCUM_STEPS
            scaler_G.scale(loss_G).backward()
            if is_last_accum:
                scaler_G.unscale_(opt_G)
                nn.utils.clip_grad_norm_(
                    filter(lambda p: p.requires_grad, generator.parameters()), GRAD_CLIP_NORM
                )
                scaler_G.step(opt_G)
                scaler_G.update()
                opt_G.zero_grad()
        else:
            fake = generator(x_in)
            loss_G = criterion(fake, gt, discriminator) / GRAD_ACCUM_STEPS
            loss_G.backward()
            if is_last_accum:
                nn.utils.clip_grad_norm_(
                    filter(lambda p: p.requires_grad, generator.parameters()), GRAD_CLIP_NORM
                )
                opt_G.step()
                opt_G.zero_grad()

        total_loss_G += loss_G.item() * GRAD_ACCUM_STEPS
        total_loss_D += loss_D.item() * GRAD_ACCUM_STEPS
        n_batches    += 1

    avg_G = total_loss_G / n_batches
    avg_D = total_loss_D / n_batches

    # ── EMA update ───────────────────────────────────────────────────
    ema_generator.update(generator)

    # ── Validation loop ───────────────────────────────────────────────
    generator.eval()
    val_loss_G = 0.0
    with torch.no_grad():
        for vbatch in val_loader:
            vperson  = vbatch["person"].to(DEVICE)
            vgarment = vbatch["garment"].to(DEVICE)
            vgt      = vbatch["gt"].to(DEVICE)
            vx_in    = torch.cat([vperson, vgarment], dim=1)
            if _use_amp:
                with torch.amp.autocast('cuda', dtype=dtype):
                    vfake    = generator(vx_in)
                    vl       = criterion(vfake, vgt, discriminator)
            else:
                vfake = generator(vx_in)
                vl    = criterion(vfake, vgt, discriminator)
            val_loss_G += vl.item()
    avg_val_G = val_loss_G / max(len(val_loader), 1)

    # ── Schedulers step ───────────────────────────────────────────────
    sched_G.step()
    sched_D.step()

    # ── History ───────────────────────────────────────────────────────
    epoch_time = time.time() - epoch_start
    history["train_G"].append(avg_G)
    history["train_D"].append(avg_D)
    history["val_G"].append(avg_val_G)
    history["epoch_times"].append(epoch_time)

    # ── Console log ───────────────────────────────────────────────────
    print(
        f"Epoch {epoch+1:4d}/{EPOCHS} | "
        f"G={avg_G:.4f}  D={avg_D:.4f}  val_G={avg_val_G:.4f} | "
        f"{epoch_time:.1f}s"
    )

    # ── Save best checkpoint ──────────────────────────────────────────
    if avg_val_G < best_val_loss:
        best_val_loss = avg_val_G
        ckpt_path = CKPT_DIR / "best_generator.pt"
        torch.save(generator.state_dict(), str(ckpt_path))
        print(f"   ✅ Best checkpoint saved (val_G={best_val_loss:.4f})")

    # ── Periodic checkpoint ───────────────────────────────────────────
    if (epoch + 1) % SAVE_EVERY == 0:
        periodic_path = CKPT_DIR / f"generator_epoch{epoch+1:04d}.pt"
        torch.save(generator.state_dict(), str(periodic_path))
        print(f"   💾 Checkpoint saved: {periodic_path.name}")

    # ── Early stopping ────────────────────────────────────────────────
    if early_stopping(avg_val_G):
        print(f"\n⏹  Early stopping triggered at epoch {epoch+1}")
        break

print(f"\n✅ Training complete — best val_G = {best_val_loss:.4f}")

---
## Step 10 — Plot Loss Curves & Save to Drive

In [ ]:
import matplotlib.pyplot as plt

# ── Loss curves ───────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

epochs_x = range(1, len(history['train_G']) + 1)
axes[0].plot(epochs_x, history['train_G'], label='Train G', color='#4c8bf5', linewidth=2)
axes[0].plot(epochs_x, history['val_G'],   label='Val G',   color='#ea4335', linewidth=2, linestyle='--')
axes[0].set_title('Generator Loss', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_x, history['train_D'], label='Train D', color='#34a853', linewidth=2)
axes[1].set_title('Discriminator Loss', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()

plot_path = Path("data/arvton/outputs/training_curves.png")
plot_path.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(str(plot_path), dpi=120, bbox_inches='tight')
plt.show()
print(f"✅ Loss curves saved to {plot_path}")

In [ ]:
# ── Copy checkpoints & logs to Google Drive ───────────────────────────
import shutil

drive_ckpt_dir = Path("/content/drive/MyDrive/ARVTON/checkpoints")
drive_ckpt_dir.mkdir(parents=True, exist_ok=True)

# Copy all .pt files
pt_files = list(CKPT_DIR.glob("*.pt"))
for pt in pt_files:
    dest = drive_ckpt_dir / pt.name
    shutil.copy2(str(pt), str(dest))
    print(f"   📤 {pt.name} → Drive")

# Copy training curves
if plot_path.exists():
    drive_log_dir = Path("/content/drive/MyDrive/ARVTON/logs")
    drive_log_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy2(str(plot_path), str(drive_log_dir / "training_curves.png"))
    print(f"   📤 training_curves.png → Drive")

print(f"\n✅ All outputs copied to Google Drive ARVTON/ folder")
print(f"   Checkpoints : /content/drive/MyDrive/ARVTON/checkpoints/")
print(f"   Logs        : /content/drive/MyDrive/ARVTON/logs/")

In [ ]:
# ── (Optional) Download best checkpoint directly ───────────────────────
from google.colab import files

best_ckpt = CKPT_DIR / "best_generator.pt"
if best_ckpt.exists():
    files.download(str(best_ckpt))
    print(f"✅ Downloading {best_ckpt.name}")
else:
    print("⚠️  No best checkpoint found — training may not have completed")